# A2: Unsupervised Learning on Time Series Data — Anomaly Detection

**Course:** Machine Learning  
**Department:** Mechanical and Industrial Engineering, NTNU  

**Dataset:** NASA Anomaly Detection Dataset — SMAP & MSL  
**Source:** [Kaggle — patrickfleith/nasa-anomaly-detection-dataset-smap-msl](https://www.kaggle.com/datasets/patrickfleith/nasa-anomaly-detection-dataset-smap-msl)  
**Original paper:** Hundman et al., 2018 — *Detecting Spacecraft Anomalies Using LSTMs and Nonparametric Dynamic Thresholding* (NASA JPL)

---
# 1. Frame the Problem and Look at the Big Picture

## 1.1 Business Objective

**Goal:** Detect anomalies in telemetry time series collected from two NASA spacecraft:
- **SMAP** (Soil Moisture Active Passive) — an Earth-orbiting satellite
- **MSL** (Mars Science Laboratory) — the Curiosity rover on Mars

In operational practice, anomaly detection on spacecraft telemetry enables ground engineers to identify unexpected system behaviour early, investigate potential faults, and prevent mission-critical failures — without requiring continuous human monitoring of hundreds of channels.

**Dataset summary:**

| Spacecraft | Channels | Total timesteps | Labeled anomalies |
|------------|----------|-----------------|-------------------|
| SMAP       | 55       | 429,735         | 69                |
| MSL        | 27       | 66,709          | 36                |

Each channel is stored as a separate `.npy` file. The first column of each file is the raw telemetry reading; remaining columns are pre-engineered command/context features. Labels (anomaly index ranges) are provided in `labeled_anomalies.csv` — **for evaluation only, not used during training**.

## 1.2 How the Solution Will Be Used

**Offline mode (this assignment):**  
Models are trained on the provided anomaly-free training split. The fitted model is then applied to the test split and flagged anomalies are compared against the ground-truth labels in `labeled_anomalies.csv`.

**Online mode (real-time spacecraft monitoring):**  
- Telemetry packets arrive continuously from the spacecraft
- Each incoming reading is appended to a sliding buffer
- The buffer is preprocessed (using the already-fitted scaler) and scored by the model
- If the anomaly score exceeds the pre-calibrated threshold, an alert is raised for the operations team
- The ground station can then cross-check the ISA (Integrated System Anomaly) reports

Online deployment considerations:
- **Latency:** Must score a new sample within the telemetry downlink cadence
- **Statefulness:** The model must be stateless or maintain a compact buffer (rules out standard DBSCAN)
- **Drift:** Spacecraft operating modes change over time — periodic retraining may be needed

## 1.3 Current Solutions / Baselines

- **Rule-based alarm systems:** NASA's existing monitoring uses fixed thresholds on individual telemetry channels — effective only for obvious out-of-range values (point anomalies)
- **Manual review:** Operations engineers periodically inspect telemetry plots and file ISA (Integrated System Anomaly) reports when anomalies are found — time-consuming and not scalable across 55+ channels
- **Statistical process control:** Simple z-score or rolling-mean-deviation alerts — misses contextual anomalies where values are individually normal but anomalous in context

The Hundman et al. (2018) paper used an LSTM + nonparametric dynamic thresholding approach. This assignment compares simpler, more interpretable unsupervised techniques against this context.

## 1.4 Problem Type

- **Unsupervised learning** — models are trained on the clean training split with no anomaly labels
- **Anomaly / novelty detection** on multivariate time series telemetry data
- Two types of anomalies present in the dataset:
  - **Point anomalies** — individual timesteps that are statistically extreme; detectable by distance- or density-based methods
  - **Contextual anomalies** — values that are normal in isolation but anomalous given surrounding context; require temporal modelling to detect

Techniques to implement and compare:
1. **K-Means** — distance to nearest centroid as anomaly score - Sila
2. **DBSCAN** — noise points (label = -1) treated as anomalies - Sila
3. **Gaussian Mixture Model (GMM)** — negative log-likelihood as anomaly score - Tomas
4. **Isolation Forest** — anomaly score from random path lengths - Sila
5. **Local Outlier Factor (LOF)** — local density deviation as anomaly score - Tomas
6. SVM - Sila
7. PCA - Tomas

## 1.5 Performance Measurement

Model performance is evaluated using multiple complementary metrics:

- **F1 score** as the primary comparison metric — balances precision and recall, appropriate for the imbalanced anomaly detection setting
- **Precision and Recall** to separately assess false alarm rate and detection coverage; in spacecraft operations, recall is prioritised (missing an anomaly is more costly than a false alert)
- **AIC / BIC** for selecting the optimal number of components in GMM
- **Silhouette score** for assessing the separation quality of K-Means clusters

Ground-truth labels from `labeled_anomalies.csv` are used exclusively for evaluation — not during training. This combination allows both statistical detection quality and operational practicality to be assessed.

## 1.6 Minimum Performance Requirements

Since ground-truth labels are available in `labeled_anomalies.csv`, we can set concrete targets for the models:

- **F1 score > 0.5** on the test set — a random classifier at ~5% contamination would score ~0.1
- **Recall > 0.6** — missing anomalies is costly in spacecraft operations; recall is prioritised over precision
- **False alarm rate < 10%** — too many false alerts would make the system impractical for operators
- Models should detect anomalies **without being told where they are** (trained only on the clean training split)

---
# 2. Libraries

In [ ]:
# Main libraries
import numpy as np
import pandas as pd

# Visualization
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns

# Preprocessing
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.decomposition import PCA

# Unsupervised / Anomaly Detection Models
from sklearn.cluster import KMeans, DBSCAN
from sklearn.mixture import GaussianMixture, BayesianGaussianMixture
from sklearn.ensemble import IsolationForest
from sklearn.neighbors import LocalOutlierFactor
from sklearn.svm import OneClassSVM

# Metrics
from sklearn.metrics import (
    silhouette_score,
    davies_bouldin_score,
    confusion_matrix,
    classification_report,
    roc_auc_score,
    precision_recall_curve
)

# General utilities
import warnings
warnings.filterwarnings('ignore')

# Reproducibility
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

# Plot style
plt.rcParams['figure.figsize'] = (14, 4)
plt.rcParams['axes.grid'] = True
sns.set_theme(style='whitegrid')

%load_ext autoreload
%autoreload 2

---
# 3. Get the Data

## 3.1 Load Dataset

**File structure:**
```
archive/
├── labeled_anomalies.csv          # ground-truth labels (chan_id, spacecraft, anomaly_sequences, class, num_values)
└── data/data/
    ├── train/
    │   ├── P-1.npy                # SMAP channel P-1 — training split (anomaly-free)
    │   ├── M-1.npy                # MSL  channel M-1 — training split
    │   └── ...                    # 82 channels total
    └── test/
        ├── P-1.npy                # SMAP channel P-1 — test split (may contain anomalies)
        ├── M-1.npy
        └── ...
```

**Each `.npy` file:** shape `(timesteps, n_features)`  
- Column 0: raw telemetry value for that channel  
- Columns 1+: pre-engineered command/context features (one-hot encoded spacecraft commands)

**Strategy:** We will work on a **single channel** (or a small set) at a time.  
Labels in `labeled_anomalies.csv` give `anomaly_sequences` as `[[start, end], ...]` — index ranges into the **test** array for that channel.

In [ ]:
import sys
import os

# Make sure Dataset.py is importable from the project directory
BASE_DIR = os.getcwd()
if BASE_DIR not in sys.path:
    sys.path.insert(0, BASE_DIR)

from Dataset import DatasetOperations

# Paths (mirrors Main.py)
train_path       = os.path.join(BASE_DIR, "archive", "data", "data", "train")
test_path        = os.path.join(BASE_DIR, "archive", "data", "data", "test")
result_file_path = os.path.join(BASE_DIR, "archive", "labeled_anomalies.csv")

dt = DatasetOperations(test_path=test_path, train_path=train_path, results_path=result_file_path)
train_dataset, test_dataset, results = dt.load_data()

## 3.2 Basic Dataset Info

Inspect the loaded arrays and the label file:
- How many timesteps in the train vs test split?
- How many features per channel?
- Which anomaly class is present — point, contextual, or both?
- What percentage of the test set is labeled anomalous?
- Presence of missing values
- check for nonbinary values 

In [ ]:
dt.dataset_info()

---
# 4. Explore the Data (EDA)

## 4.1 Time Series Visualization

Plot the raw telemetry signal (column 0) for the selected channel(s) over time.  
Overlay the known anomaly windows from `labeled_anomalies.csv` as shaded regions on the **test** portion — this gives a visual intuition of what we are trying to detect.

In [ ]:
dt.plot_data(choosen_dataset="training_dataset", start_channel_id="A-1")

## 4.2 Correlation Matrix 
corelation check function creates correlation matrix and creates a csv report about correlation between files

In [ ]:
correlation_outfile_path = BASE_DIR
correlation_dictionary = dt.correlation_check(mode="training_dataset",correlation_csv_report = False,correlation_outfile_path=correlation_outfile_path,corr_calc_method="spearman")

## 4.5 Anomaly Label Analysis

Inspect the ground-truth labels from `labeled_anomalies.csv`:
- How many channels have **point** vs **contextual** anomalies?
- What is the average anomaly window length?
- What fraction of each channel's test set is anomalous?

This informs the expected contamination rate and helps set the threshold for each model.

In [ ]:
import ast

# Point vs contextual breakdown
print("=== Anomaly Type Breakdown ===")
print(results['class'].value_counts().to_string())

# Anomaly fraction per channel (using test set lengths from test_dataset)
print("\n=== Anomaly Rate per Channel ===")
rows = []
for _, row in results.iterrows():
    cid = row['chan_id']
    if cid not in test_dataset:
        continue
    n_total = len(test_dataset[cid])
    seqs    = ast.literal_eval(row['anomaly_sequences'])
    n_anom  = sum(end - start + 1 for start, end in seqs)
    avg_len = np.mean([end - start + 1 for start, end in seqs]) if seqs else 0
    rows.append({'channel': cid, 'class': row['class'],
                 'anomaly_%': round(100 * n_anom / n_total, 1),
                 'avg_seq_len': round(avg_len, 1)})

df_label = pd.DataFrame(rows).sort_values('anomaly_%', ascending=False)
print(df_label.to_string(index=False))
print(f"\nOverall avg anomaly rate : {df_label['anomaly_%'].mean():.1f}%")
print(f"Overall avg sequence len : {df_label['avg_seq_len'].mean():.0f} timesteps")

---
# 5. Prepare the Data

## 5.1 Train / Test Split 

The NASA dataset **already provides a pre-defined chronological train/test split** — no splitting needed:

- `archive/data/data/train/{chan_id}.npy` → training data (anomaly-free, used for fitting models)
- `archive/data/data/test/{chan_id}.npy` → test data (may contain anomalies, evaluated against labels)

Ground-truth labels for the **test set only** are in `labeled_anomalies.csv`.  
The training set is considered clean — this is the standard assumption for unsupervised anomaly detection (train on normal, detect deviations at test time).
First round of method usage will be perform on smaller dataset made up of 10 randomly selected files
Also to lower amount of data function sort_by_corr creates a groups of files with high correlation

In [ ]:
import ast

CHANNEL_ID = 'D-9'

# Ground-truth label vector for single-channel plots
row    = results[results['chan_id'] == CHANNEL_ID]
y_test = np.zeros(len(test_dataset[CHANNEL_ID]), dtype=int)
if not row.empty:
    for start, end in ast.literal_eval(row.iloc[0]['anomaly_sequences']):
        y_test[start:end + 1] = 1

print(f"Channel      : {CHANNEL_ID}")
print(f"Train shape  : {train_dataset[CHANNEL_ID].shape}")
print(f"Test shape   : {test_dataset[CHANNEL_ID].shape}")
print(f"Anomaly class: {row.iloc[0]['class']}")
print(f"Anomalies    : {y_test.sum()} / {len(y_test)}  ({100*y_test.mean():.1f}%)")

# 10-channel subset for model training and evaluation (same seed as Tomas)
train_subset, test_subset = dt.select_subset(
    random_selection=True, manual_file_names=None, subset_size=10, seed=42
)
print(f"\nSubset channels: {list(train_subset.keys())}")

## 5.2 Feature Engineering

The `.npy` files already include pre-engineered command features alongside the raw telemetry. Options:

- **Use as-is:** All columns (raw telemetry + command features)
- **Telemetry only:** Column 0 only — simplest, but loses command context
- **Rolling statistics:** Add sliding-window mean and std of the telemetry value to capture local temporal behaviour — especially useful for detecting **contextual anomalies**
- **Lag features:** Append previous timestep values to give models short-term memory

In [ ]:
from TimeContext import TimeContextModif

# Apply rolling statistics to the 10-channel subset
# D-9 is in the subset so train_fe[CHANNEL_ID] works for both visualization and evaluation
tc = TimeContextModif(train_dataset=train_subset, test_dataset=test_subset)
train_fe, test_fe = tc.add_rolling_statistics(window_length=200)

n_orig     = train_dataset[CHANNEL_ID].shape[1]
n_enriched = train_fe[CHANNEL_ID].shape[1]
print(f"Features  :  {n_orig}  →  {n_enriched}  (original + rolling mean/std/min/max)")

## 5.3 Preprocessing Pipeline

In [ ]:
from Evaluation import EvaluateResults

# Point-adjust evaluation — standard for time series anomaly detection:
# if ANY point in an anomaly window is detected, the whole window counts as found.
# This is how the research community evaluates on the NASA SMAP/MSL dataset.
evaluation = EvaluateResults()
evaluation.load_solution(results)

print("Evaluation ready.")

## 5.4 Dimensionality Reduction — PCA (for visualization)

In [ ]:
from sklearn.decomposition import PCA

pca        = PCA(n_components=2, random_state=42)
X_train_2d = pca.fit_transform(train_fe[CHANNEL_ID])

ev = pca.explained_variance_ratio_
print(f"Explained variance: PC1 = {ev[0]:.1%},  PC2 = {ev[1]:.1%}  (total {ev.sum():.1%})")

plt.figure(figsize=(8, 5))
plt.scatter(X_train_2d[:, 0], X_train_2d[:, 1], s=1, alpha=0.4, color='steelblue')
plt.title(f'PCA — Training Data  ({CHANNEL_ID})')
plt.xlabel('PC1')
plt.ylabel('PC2')
plt.tight_layout()
plt.show()

---
# 6. Explore Many Different Models and Shortlist the Best Ones

## 6.1 Helper Functions

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report, f1_score

def plot_results(y_true, y_pred, model_name='Model'):
    """
    Show a confusion matrix heatmap + one-line summary.
    Much easier to read than classification_report.
    """
    cm = confusion_matrix(y_true, y_pred)
    tn, fp, fn, tp = cm.ravel()
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall    = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1        = f1_score(y_true, y_pred, zero_division=0)

    # --- Heatmap ---
    fig, ax = plt.subplots(figsize=(4, 3))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=['Normal', 'Anomaly'],
                yticklabels=['Normal', 'Anomaly'])
    ax.set_xlabel('Predicted')
    ax.set_ylabel('Actual')
    ax.set_title(f'{model_name}')
    plt.tight_layout()
    plt.show()

    # --- Summary line ---
    print(f"  Caught  : {tp} of {tp+fn} anomalies  (Recall  = {recall:.1%})")
    print(f"  False alarms: {fp}  (Precision = {precision:.1%})")
    print(f"  F1 Score: {f1:.3f}")

## 6.2 Model 1: K-Means Clustering

In [ ]:
from KMeans import KMeansAnalyzer

# Elbow plot on D-9 to choose K
km_analyzer = KMeansAnalyzer(train_fe, test_fe)
km_analyzer.elbow_plot(CHANNEL_ID, k_range=range(2, 12))

In [ ]:
K_BEST = 5

km_analyzer.fit_all(k=K_BEST)
km_analyzer.plot_clusters(CHANNEL_ID, X_train_2d)

train_km, test_km = km_analyzer.get_enriched_features()

km_preds  = km_analyzer.get_batch_predictions(threshold_percentile=95)
km_report = evaluation.compare_methods_results(predictions_dict=km_preds)
evaluation.plot_hits_vs_misses(km_report, title='K-Means — Centroid Distance')
print(km_report[['Channel', 'Precision', 'Recall', 'F1_Score']].to_string(index=False))

**K-Means Analysis**

The elbow plot has no clear bend, it just keeps going down. This means there is no obvious "right" number of clusters for this channel, so K=5 is a reasonable guess.

The results look good at first. Recall is 1.0 everywhere, meaning it found all the anomalies. But the reason is not good: the model flagged so many points that it hit every anomaly window by accident. G-1 at 1.4% precision means for every real anomaly it found, it also flagged 70 normal points as anomalies.

D-9 and D-15 work better (precision 0.78 and 0.60) because in those channels the anomalies happen to be far away from the normal groups. In G-1, T-5 and T-3 they are not, so the anomalies look just like normal data to K-Means.

The useful part here is not the anomaly detection but the 125 to 130 features output. Those 5 extra columns (centroid distances) get used by Isolation Forest and SVM in the next sections.

## 6.3 Model 2: DBSCAN

**DBSCAN** (Density-Based Spatial Clustering of Applications with Noise) groups points that are close together into clusters and marks points in low-density regions as **noise** (label = -1). These noise points are our anomalies — no separate threshold is needed.

**Key parameters:**
- `eps` — maximum distance between two points to be considered neighbors. Found using the **k-distance plot**: sort all points by their distance to the k-th nearest neighbor and look for the "knee" in the curve.
- `min_samples` — minimum number of neighbors to form a dense cluster.

**Why it works here:** anomalous telemetry points tend to be genuinely isolated in feature space — no dense neighborhood. DBSCAN detects this structurally, without assuming any data distribution.

In [ ]:
from DBSCANModel import DBSCANModel

# K-distance plot: look for the 'knee' — that y-value is the natural eps for this channel
db_analyzer = DBSCANModel(train_fe, test_fe)
db_analyzer.k_distance_plot(CHANNEL_ID, k=5)

In [ ]:
EPS         = 2.0
MIN_SAMPLES = 10

db_analyzer = DBSCANModel(train_fe, test_fe)
db_analyzer.fit_all(eps=EPS, min_samples=MIN_SAMPLES)
db_preds   = db_analyzer.get_batch_predictions()
db_report  = evaluation.compare_methods_results(predictions_dict=db_preds)
evaluation.plot_hits_vs_misses(db_report, title='DBSCAN')
print(db_report[['Channel', 'Precision', 'Recall', 'F1_Score']].to_string(index=False))

**DBSCAN Analysis**

The k-distance plot for D-9 shows a sharp drop in the first ~50 points, then goes flat near zero. The bend is around 1.0 to 1.5, so eps=2.0 is slightly above it, meaning the model is being a bit generous about what counts as a neighbor.

Macro F1 = 0.69, which is a big improvement over K-Means. Most channels work well. D-9 is almost perfect (F1=0.999), D-3 (0.974), D-15 (0.887). These channels have clear groups in the data, and the anomalies sit outside those groups as isolated points.

The two failures are G-7 and T-5 (F1=0.0 for both). G-7 got 1 cluster and 0 noise points, meaning eps=2.0 is too large for this channel so everything gets absorbed into one big cluster and nothing is flagged. T-5 found 41 noise points but none of them matched the real anomalies.

This is the main weakness of DBSCAN: one eps value for all channels does not work well. Each channel has different data so the right eps is different for each one. G-7 needs a smaller eps, but making it smaller for all channels would break the others.

## 6.4 Model 3: Gaussian Mixture Model (GMM)

In [ ]:
# --- Select number of components using BIC / AIC ---
# bic_scores, aic_scores = [], []
# N_range = range(1, 12)
# for n in N_range:
#     gmm = GaussianMixture(n_components=n, random_state=RANDOM_SEED)
#     gmm.fit(X_train)
#     bic_scores.append(gmm.bic(X_train))
#     aic_scores.append(gmm.aic(X_train))

# plt.plot(N_range, bic_scores, marker='o', label='BIC')
# plt.plot(N_range, aic_scores, marker='s', label='AIC')
# plt.xlabel('Number of components')
# plt.ylabel('Score')
# plt.title('GMM Model Selection — BIC/AIC')
# plt.legend()
# plt.show()

In [ ]:
N_COMPONENTS_GMM = 3  # replace after BIC/AIC inspection

# gmm = GaussianMixture(n_components=N_COMPONENTS_GMM, covariance_type='full', random_state=RANDOM_SEED)
# gmm.fit(X_train)

# Anomaly score = negative log-likelihood
# log_prob_train = gmm.score_samples(X_train)
# log_prob_test  = gmm.score_samples(X_test)

# threshold_gmm = np.percentile(log_prob_train, 5)  # low density = anomaly
# anomalies_gmm = (log_prob_test < threshold_gmm).astype(int)

# plot_anomaly_scores(-log_prob_test, title='GMM Anomaly Scores (−log p)', threshold=-threshold_gmm)
# print(f'GMM flagged {anomalies_gmm.sum()} anomalies ({100*anomalies_gmm.mean():.1f}%)')

## 6.5 Model 4: Isolation Forest

**Isolation Forest** detects anomalies by randomly partitioning the feature space using decision trees. Anomalous points are **easier to isolate** — they require fewer splits to be separated from the rest, resulting in a shorter average path length through the trees.

**Key parameters:**
- `contamination` — expected fraction of anomalies in the data. Used to set the decision threshold. Set to 0.10 based on parameter sweep (domain estimate, not from labels — using labels would be data leakage).
- `n_estimators` — number of isolation trees (100 is standard).

**Why it works here:** point anomalies in telemetry tend to sit in sparse regions of feature space — exactly what isolation trees are designed to find. Works on the **K-Means enriched features** (`train_km`/`test_km`) for better separation of operating modes.

In [ ]:
from IsolationForestModel import IsolationForestModel

CONTAMINATION = 0.10

iso = IsolationForestModel(train_km, test_km)
iso.fit_all(contamination=CONTAMINATION)
iso_preds   = iso.get_batch_predictions()
iso_report  = evaluation.compare_methods_results(predictions_dict=iso_preds)
evaluation.plot_hits_vs_misses(iso_report, title='Isolation Forest')
print(iso_report[['Channel', 'Precision', 'Recall', 'F1_Score']].to_string(index=False))

**Isolation Forest Analysis**

Macro F1 = 0.43, better than K-Means but worse than DBSCAN.

Every channel has Recall = 1.0 again, which means the same problem as K-Means: the model flags too many points and accidentally hits every anomaly window. The bar chart shows big red bars on almost every channel.

D-3 works well (F1=0.92) and D-9 is decent (F1=0.84). These channels have enough anomalous points that even with extra false alarms the precision stays reasonable. G-1 (0.05 precision) and E-11 (0.08 precision) are the worst, meaning the model flags thousands of normal points as anomalies.

The number of features used per channel is interesting. G-7 only has 9 usable features after removing constant columns, while D-3 has 82. G-7's poor precision (0.31) might be partly because there is not enough information for the model to tell normal from anomalous.

The K-Means enriched features help here compared to using raw data, since the centroid distances give the model extra context about operating modes. But Isolation Forest still looks at each point in time on its own, so it cannot pick up on patterns that only appear over time.

## 6.6 Model 5: Local Outlier Factor (LOF)

In [ ]:
# lof = LocalOutlierFactor(n_neighbors=20, contamination=CONTAMINATION, novelty=True)
# lof.fit(X_train)

# scores_lof = lof.decision_function(X_test)
# preds_lof  = lof.predict(X_test)
# anomalies_lof = (preds_lof == -1).astype(int)

# plot_anomaly_scores(-scores_lof, title='LOF Anomaly Scores')
# print(f'LOF flagged {anomalies_lof.sum()} anomalies ({100*anomalies_lof.mean():.1f}%)')

## 6.7 Model 6: One-Class SVM

**One-Class SVM** learns a decision boundary around the normal training data in a high-dimensional kernel space (RBF by default). Points that fall **outside** the boundary at test time are classified as anomalies.

**Key parameters:**
- `nu` — upper bound on the fraction of training errors and lower bound on support vectors. Acts similarly to contamination — set to 0.05.
- `kernel` — `rbf` (default) maps data into infinite-dimensional space, allowing complex non-linear boundaries.

**Limitation:** One-Class SVM is sensitive to the scale and dimensionality of the data. With 25–100 features, the RBF kernel boundary can become poorly defined, leading to over- or under-detection. It is generally the weakest of the four models on this dataset.

In [ ]:
from OneClassSVMModel import OneClassSVMModel

NU = 0.05

svm = OneClassSVMModel(train_km, test_km)
svm.fit_all(nu=NU)
svm_preds   = svm.get_batch_predictions()
svm_report  = evaluation.compare_methods_results(predictions_dict=svm_preds)
evaluation.plot_hits_vs_misses(svm_report, title='One-Class SVM')
print(svm_report[['Channel', 'Precision', 'Recall', 'F1_Score']].to_string(index=False))

**One-Class SVM Analysis**

Macro F1 = 0.22, the worst result of all the models we tested.

The pattern is the same as K-Means and Isolation Forest: Recall = 1.0 everywhere because it flags so many points that it eventually hits every real anomaly window. But the precision is terrible. T-3 has 8395 false positives and G-1 has 8348, meaning almost everything it flags is wrong.

D-3 is the only channel where it works reasonably (F1=0.57), probably because D-3 has a large anomaly region that is hard to miss. T-5 is the worst at F1=0.028. It found the anomaly but with so many false alarms that precision is near zero.

The problem is the feature space. With 25 to 130 features per channel, the RBF kernel has trouble drawing a meaningful boundary around normal data. In high dimensions, almost everything ends up looking like it is on the boundary, so the model flags huge chunks of normal data as anomalies.

nu=0.05 means the model expects about 5% of training points to be on the wrong side of the boundary, but in practice it flags way more than 5% of the test data. Tuning nu lower would help precision but hurt recall.

Overall, One-Class SVM is not a good fit for this dataset. The high dimensionality and temporal nature of the data work against it.

## 6.8 Model 7: LSTM Autoencoder

**LSTM Autoencoder** is a neural network that learns to compress and reconstruct normal time series sequences. It is trained only on normal (training) data, so it learns what "normal" looks like over time. At test time, sequences that deviate from learned patterns produce a high **reconstruction error (MSE)** — these are flagged as anomalies.

**Architecture:**
- **Encoder** — LSTM reads a window of 50 timesteps and compresses it into a single hidden vector
- **Decoder** — another LSTM takes that vector and reconstructs the original sequence
- **Threshold** — windows with MSE above the 99th percentile of all test scores are anomalies

**Why it outperforms static models on this dataset:**  
Static models (IF, SVM, DBSCAN) look at each timestep in isolation — they ask *"does this point look unusual?"*  
The LSTM asks *"does this sequence of 50 timesteps look unusual?"* — which is exactly what contextual anomalies require.

**Note on feature selection:**  
Using all 25 features degraded performance (Macro F1: 0.787 → 0.617) because a single shared model cannot simultaneously learn the reconstruction patterns of 10 channels with different sensor distributions. Column 0 provides a consistent primary telemetry signal that generalises across channels.

In [ ]:
from LSTMAutoencoder import LSTM_AE_Detector

lstm = LSTM_AE_Detector(seq_len=50, hidden_dim=32, epochs=5, percentile=99, n_features=1)
lstm.fit(train_subset)
lstm_preds   = lstm.prediction(test_subset)
lstm_report  = evaluation.compare_methods_results(predictions_dict=lstm_preds)
evaluation.plot_hits_vs_misses(lstm_report, title='LSTM Autoencoder')
print(lstm_report[['Channel', 'Precision', 'Recall', 'F1_Score']].to_string(index=False))

**LSTM Autoencoder Analysis**

Macro F1 = 0.93, by far the best result of all models. The bar chart is almost entirely green, which is exactly what we want.

Three channels hit perfect F1=1.0 (D-3, D-9, P-2) and D-15 is nearly perfect at 0.998. The model flagged only the right points on those channels with very low false alarms. T-4, T-3, and G-7 are also above 0.95, which is strong.

The two weaker channels are G-1 (F1=0.74) and T-5 (F1=0.78). Both have Recall=1.0, meaning all real anomalies were found, but with some extra false positives. T-5 is a small channel with only 26 real anomaly points, so 15 extra false alarms bring the precision down noticeably. G-1 similarly has a small anomaly region (121 real points) and 83 false positives. These are not failures, just harder channels.

The key reason LSTM beats every other model is that it looks at sequences of 50 timesteps together instead of single points. A normal reading that happens right after a string of unusual readings looks suspicious to the LSTM but looks perfectly fine to K-Means, DBSCAN, or SVM. That temporal memory is what makes the difference.

Training on column 0 only (the raw telemetry signal) also worked better than using all 25 features. One model cannot learn the different normal patterns of 10 separate channels at the same time. By focusing on the same type of signal across all channels, the model builds a general idea of what normal telemetry looks like.

---
# 7. Fine-Tune the Models and Combine into a Great Solution

## 7.1 Model Comparison Summary

In [ ]:
def macro_f1(report_df):
    return round(report_df['F1_Score'].mean(), 4) if not report_df.empty else None

comparison = pd.DataFrame([
    {'Model': 'K-Means',           'Macro F1': macro_f1(km_report),   'Notes': 'threshold_percentile=95, K=5'},
    {'Model': 'DBSCAN',            'Macro F1': macro_f1(db_report),   'Notes': 'eps=2.0, min_samples=10'},
    {'Model': 'Isolation Forest',  'Macro F1': macro_f1(iso_report),  'Notes': 'contamination=0.10'},
    {'Model': 'One-Class SVM',     'Macro F1': macro_f1(svm_report),  'Notes': 'nu=0.05, kernel=rbf'},
    {'Model': 'LSTM Autoencoder',  'Macro F1': macro_f1(lstm_report), 'Notes': 'seq_len=50, hidden_dim=32, percentile=99'},
])

print(comparison.to_string(index=False))

# Bar chart
fig, ax = plt.subplots(figsize=(9, 4))
colors = ['#d62728' if v < 0.5 else '#ff7f0e' if v < 0.7 else '#2ca02c'
          for v in comparison['Macro F1']]
bars = ax.barh(comparison['Model'], comparison['Macro F1'], color=colors, edgecolor='white')
ax.bar_label(bars, fmt='%.4f', padding=4, fontsize=10)
ax.set_xlim(0, 1.05)
ax.set_xlabel('Macro F1 Score (10-channel subset, point-adjust)')
ax.set_title('Model Comparison — Macro F1', fontsize=13, fontweight='bold')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.show()

## 7.2 Threshold Sensitivity Analysis

In [ ]:
# How does anomaly rate change with threshold percentile?
# thresholds = np.percentile(distances_train, np.arange(90, 100, 0.5))
# anomaly_rates = [(distances_test > t).mean() for t in thresholds]

# plt.plot(np.arange(90, 100, 0.5), np.array(anomaly_rates)*100)
# plt.xlabel('Training percentile threshold')
# plt.ylabel('Anomaly rate on test (%)')
# plt.title('K-Means Threshold Sensitivity')
# plt.show()

## 7.3 Ensemble / Best Model Selection

In [ ]:
# Majority voting ensemble
# votes = np.stack([anomalies_kmeans, anomalies_gmm, anomalies_if, anomalies_lof], axis=1)
# anomalies_ensemble = (votes.sum(axis=1) >= 2).astype(int)
# print(f'Ensemble flagged {anomalies_ensemble.sum()} anomalies ({100*anomalies_ensemble.mean():.1f}%)')

## 7.4 Visualise Final Anomalies on Raw Signal

In [ ]:
# Overlay anomaly flags on the original time series
# fig, ax = plt.subplots(figsize=(16, 4))
# ax.plot(df_test.index, df_test.iloc[:, 0], linewidth=0.7, label='Signal')
# anomaly_idx = df_test.index[anomalies_ensemble == 1]
# ax.scatter(anomaly_idx, df_test.iloc[:, 0].loc[anomaly_idx],
#            color='red', s=10, zorder=5, label='Flagged anomaly')
# ax.set_title('Detected Anomalies on Test Set')
# ax.legend()
# plt.tight_layout()
# plt.show()

---
# 8. Online Deployment Discussion

### How to Use the Solution in Online (Real-Time) Mode

In a real NASA ground operations context, telemetry data streams in continuously from the spacecraft. Here is how the trained model would be integrated:

| Aspect | Approach |
|--------|----------|
| **Data ingestion** | Each incoming telemetry packet is appended to a rolling buffer of fixed length |
| **Preprocessing** | The buffer is transformed using the already-fitted `StandardScaler` (fitted on training data) |
| **Inference** | The preprocessed buffer is scored by the model (e.g., GMM log-likelihood or Isolation Forest score) |
| **Decision** | If the score exceeds the pre-calibrated threshold, the timestep is flagged as anomalous |
| **Alert** | An alert is forwarded to the spacecraft operations team for review against ISA reports |
| **Model refresh** | Periodically retrain on a recent window of confirmed-normal telemetry to account for mission phase changes |

**Model suitability for online use:**

| Model | Online-compatible? | Notes |
|-------|--------------------|-------|
| K-Means | Yes | Store centroids; compute distance per new sample |
| DBSCAN | No (without adaptation) | Requires access to full dataset at inference time |
| GMM | Yes | Evaluate log-likelihood per new sample — very fast |
| Isolation Forest | Yes | Score each sample independently after training |
| LOF | Yes (with `novelty=True`) | Requires storing training data for neighbour queries |

**Recommended online pipeline:**  
`Telemetry stream → Rolling buffer → StandardScaler → GMM or Isolation Forest → Threshold → Alert`

---
# 9. ML Project Checklist Report

### Frame the Problem
- [ ] Business objective defined (spacecraft anomaly detection — SMAP & MSL)
- [ ] Problem type identified (unsupervised anomaly detection on multivariate time series)
- [ ] Performance metrics selected (F1, Precision, Recall against labeled_anomalies.csv)

### Get the Data
- [ ] Dataset loaded from `archive/data/data/train/` and `archive/data/data/test/`
- [ ] Ground-truth labels loaded from `archive/labeled_anomalies.csv`
- [ ] Channel selection documented (which channels are analysed and why)

### Explore the Data
- [ ] Raw telemetry signals plotted with ground-truth anomaly windows overlaid
- [ ] Missing values checked
- [ ] Distributions inspected
- [ ] Correlations analysed
- [ ] Anomaly label breakdown analysed (point vs contextual, anomaly rate per channel)

### Prepare the Data
- [ ] Pre-defined train/test split confirmed (no manual splitting needed)
- [ ] Preprocessing pipeline fitted on training data only
- [ ] Feature engineering documented (rolling stats, lag features, or raw)
- [ ] Ground-truth label arrays constructed for the test set
- [ ] No data leakage from test to train confirmed

### Model Exploration
- [ ] K-Means implemented (elbow method for K, centroid distance as score)
- [ ] DBSCAN implemented (k-distance plot for eps)
- [ ] GMM implemented (BIC/AIC for n_components, log-likelihood as score)
- [ ] Isolation Forest implemented
- [ ] LOF implemented
- [ ] All models evaluated on same channel(s) using F1, Precision, Recall

### Fine-Tune
- [ ] Hyperparameters tuned for best models
- [ ] Threshold sensitivity analysed
- [ ] Ensemble / best model selected and justified
- [ ] Final anomalies visualised overlaid on raw telemetry

### Online Deployment
- [ ] Online pipeline described (buffer → scaler → model → threshold → alert)
- [ ] Model-by-model online suitability assessed
- [ ] Concept drift / model refresh strategy outlined

---
### Summary

**Best performing model:** [TBD]  
**Why it outperforms others:** [TBD — discuss point vs contextual anomalies, feature space geometry, etc.]  
**Recommended online model:** [TBD]  
**Key anomalies detected:** [TBD — describe what was found and on which channels]